In [0]:
dbutils.widgets.text("catalog", "claims", "Catalog Name")
catalog = dbutils.widgets.get("catalog")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.gov.admin_users') (
    user_name STRING
)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.gov.user_states') (
    user_name STRING,
    state STRING
)

In [0]:
%sql
CREATE OR REPLACE FUNCTION IDENTIFIER(:catalog || '.gov.mask_last4')(v STRING)
RETURNS STRING
RETURN
    CASE
        WHEN EXISTS (
            SELECT 1
            FROM claims.gov.admin_users
            WHERE LOWER(user_name) = LOWER(SESSION_USER())
        )
        THEN v
        ELSE CONCAT('****', RIGHT(v, 4))
    END;

In [0]:
%sql
-- Mask national ID
ALTER TABLE claims.raw.members
ALTER COLUMN national_id
SET MASK claims.gov.mask_last4;

In [0]:
%sql
-- Mask phone number
ALTER TABLE claims.raw.members
ALTER COLUMN phone
SET MASK claims.gov.mask_last4;

In [0]:
%sql
INSERT INTO IDENTIFIER(:catalog || '.gov.admin_users') VALUES (
    SESSION_USER
)

In [0]:
%sql
CREATE OR REPLACE FUNCTION
IDENTIFIER(:catalog || '.gov.claims_state_filter')(
    p_hospital_id STRING
)
RETURNS BOOLEAN

RETURN
    CASE
        -- Administrator can see all claims
        WHEN EXISTS (
            SELECT 1
            FROM IDENTIFIER(:catalog || '.gov.admin_users')
            WHERE LOWER(user_name) = LOWER(SESSION_USER())
        )
        THEN TRUE

        -- Check user's assigned states against hospital state
        ELSE EXISTS (
            SELECT 1
            FROM IDENTIFIER(:catalog || '.gov.user_states') us

            INNER JOIN IDENTIFIER(:catalog || '.raw.hospitals') h
                ON LOWER(TRIM(us.state)) = LOWER(TRIM(h.state))

            WHERE LOWER(us.user_name) = LOWER(SESSION_USER())
              AND h.hospital_id = p_hospital_id
        )
    END;

In [0]:
%sql
ALTER MATERIALIZED VIEW claims.gold.claims_enriched
SET ROW FILTER claims.gov.claims_state_filter
ON (hospital_id);

In [0]:
%sql
INSERT INTO IDENTIFIER(:catalog || '.gov.user_states') VALUES (
    SESSION_USER,
    "Karnataka"
),
(
    SESSION_USER,
    "Telangana"
)

In [0]:
%sql
-- Allow all account users to use the claims catalog
GRANT USE CATALOG
ON CATALOG claims
TO `account users`;

-- Allow all account users to use the Gold schema
GRANT USE SCHEMA
ON SCHEMA claims.gold
TO `account users`;

-- Allow all account users to read objects in Gold
GRANT SELECT
ON SCHEMA claims.gold
TO `account users`;

-- Allow all account users to use the Raw schema
GRANT USE SCHEMA
ON SCHEMA claims.raw
TO `account users`;

In [0]:
%sql
-- Allow reading plans
GRANT SELECT
ON TABLE claims.raw.plans
TO `account users`;

-- Allow reading hospitals
GRANT SELECT
ON TABLE claims.raw.hospitals
TO `account users`;

In [0]:
catalog = "claims"

objects = [
    ("CATALOG", catalog),
    ("SCHEMA", f"{catalog}.gold"),
    ("SCHEMA", f"{catalog}.raw"),
    ("TABLE", f"{catalog}.raw.plans"),
    ("TABLE", f"{catalog}.raw.hospitals")
]

grants = []

for object_type, object_key in objects:

    if object_type == "CATALOG":
        query = f"SHOW GRANTS ON CATALOG {object_key}"
    elif object_type == "SCHEMA":
        query = f"SHOW GRANTS ON SCHEMA {object_key}"
    else:
        query = f"SHOW GRANTS ON TABLE {object_key}"

    df = spark.sql(query)

    for row in df.collect():
        grants.append((
            object_key,
            row["Principal"],
            row["ActionType"],
            object_type,
            object_key
        ))

snapshot_df = spark.createDataFrame(
    grants,
    ["securable", "principal", "action_type", "object_type", "object_key"]
)

snapshot_df.write.mode("overwrite").saveAsTable(
    "claims.gov.grants_snapshot"
)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.gov.proof_members_masked') AS
select * from claims.raw.members

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.gov.proof_members_admin') AS
select * from claims.raw.members

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.gov.proof_claims_admin') AS
select * from claims.gold.claims_enriched

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.gov.proof_claims_unmapped') AS
select * from claims.gold.claims_enriched

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.gov.proof_claims_mapped') AS
select * from claims.gold.claims_enriched

In [0]:
%sql
ALTER TABLE claims.raw.members
ALTER COLUMN name
SET TAGS ('pii' = 'true');

ALTER TABLE claims.raw.members
ALTER COLUMN dob
SET TAGS ('pii' = 'true');

ALTER TABLE claims.raw.members
ALTER COLUMN national_id
SET TAGS ('pii' = 'true');

ALTER TABLE claims.raw.members
ALTER COLUMN phone
SET TAGS ('pii' = 'true');

ALTER TABLE claims.raw.members
ALTER COLUMN email
SET TAGS ('pii' = 'true');

In [0]:
%sql
SELECT
    column_name,
    tag_name,
    tag_value
FROM claims.information_schema.column_tags
WHERE table_name = 'members'
  AND schema_name = 'raw';

## Add comments to `claims.gold` tables and columns

Applies Unity Catalog table and column comments to every object in `claims.gold` — `claims_enriched`, `hospital_monthly`, `hospital_monthly_designer`, `hospital_monthly_vdp` — so business meaning surfaces in Catalog Explorer and `information_schema`. All four objects are materialized views published by the Claims_Pipeline.

In [0]:
%sql
-- Table and column documentation for claims.gold.claims_enriched
COMMENT ON TABLE claims.gold.claims_enriched IS
  'Gold claims fact (materialized view): one row per valid claim, enriched with point-in-time policy details from the policy SCD2 history, plan, hospital, and network-at-admission attributes, plus derived business metrics. Rows are filtered by claims.gov.claims_state_filter on hospital_id.';

COMMENT ON COLUMN claims.gold.claims_enriched.claim_id IS 'Unique identifier of the insurance claim; one row per claim.';
COMMENT ON COLUMN claims.gold.claims_enriched.member_id IS 'Identifier of the member who filed the claim; references claims.raw.members.';
COMMENT ON COLUMN claims.gold.claims_enriched.policy_id IS 'Identifier of the policy under which the claim was filed (format POL-xxxx).';
COMMENT ON COLUMN claims.gold.claims_enriched.hospital_id IS 'Identifier of the hospital where the member was admitted; references claims.raw.hospitals. Input column for the claims.gov.claims_state_filter row filter.';
COMMENT ON COLUMN claims.gold.claims_enriched.admission_date IS 'Date the member was admitted to the hospital.';
COMMENT ON COLUMN claims.gold.claims_enriched.discharge_date IS 'Date the member was discharged; on or after admission_date.';
COMMENT ON COLUMN claims.gold.claims_enriched.diagnosis_code IS 'Primary diagnosis code recorded for the admission.';
COMMENT ON COLUMN claims.gold.claims_enriched.procedure_code IS 'Primary procedure code recorded for the admission.';
COMMENT ON COLUMN claims.gold.claims_enriched.is_pre_existing IS 'TRUE if the claim relates to a pre-existing condition.';
COMMENT ON COLUMN claims.gold.claims_enriched.claim_type IS 'Category of the claim (e.g., cashless, reimbursement).';
COMMENT ON COLUMN claims.gold.claims_enriched.billed_amount IS 'Total amount billed by the hospital for the claim, in INR; always greater than 0.';
COMMENT ON COLUMN claims.gold.claims_enriched.room_rent_per_day IS 'Daily room rent charged by the hospital, in INR.';
COMMENT ON COLUMN claims.gold.claims_enriched.room_days IS 'Number of room days charged; equals datediff(discharge_date, admission_date).';
COMMENT ON COLUMN claims.gold.claims_enriched.status IS 'Processing status of the claim: approved, settled, or rejected.';
COMMENT ON COLUMN claims.gold.claims_enriched.approved_amount IS 'Amount approved for payment, in INR; between 0 and billed_amount, and 0 for rejected claims.';
COMMENT ON COLUMN claims.gold.claims_enriched.rejection_code IS 'Reason code populated when the claim is rejected; NULL otherwise.';
COMMENT ON COLUMN claims.gold.claims_enriched.submitted_ts IS 'Timestamp when the claim was submitted for processing; normally on or after admission_date.';
COMMENT ON COLUMN claims.gold.claims_enriched.settled_ts IS 'Timestamp when the claim was settled; present only for settled claims, on or after discharge.';
COMMENT ON COLUMN claims.gold.claims_enriched.channel IS 'Submission channel of the claim (e.g., online, TPA, agent).';
COMMENT ON COLUMN claims.gold.claims_enriched.plan_code_at_admission IS 'Plan code of the policy in effect on admission_date, resolved point-in-time from claims.silver.policy_scd2.';
COMMENT ON COLUMN claims.gold.claims_enriched.plan_family IS 'Family of the plan in effect at admission; from claims.raw.plans.';
COMMENT ON COLUMN claims.gold.claims_enriched.policy_start_date IS 'Start date of the policy version in effect at admission; from claims.silver.policy_scd2.';
COMMENT ON COLUMN claims.gold.claims_enriched.room_rent_cap_per_day IS 'Daily room rent cap of the plan in effect at admission, in INR; from claims.raw.plans.';
COMMENT ON COLUMN claims.gold.claims_enriched.hospital_city IS 'City of the admitting hospital; from claims.raw.hospitals.';
COMMENT ON COLUMN claims.gold.claims_enriched.hospital_city_tier IS 'City tier of the admitting hospital; from claims.raw.hospitals.';
COMMENT ON COLUMN claims.gold.claims_enriched.is_network_at_admission IS 'TRUE if the hospital was in the policy network on admission_date; from claims.raw.hospital_network_history.';
COMMENT ON COLUMN claims.gold.claims_enriched.admission_month IS 'First day of the month of admission_date (yyyy-MM-01); grouping key for monthly reporting.';
COMMENT ON COLUMN claims.gold.claims_enriched.policy_tenure_months IS 'Months between policy_start_date and admission_date.';
COMMENT ON COLUMN claims.gold.claims_enriched.length_of_stay IS 'Days between admission_date and discharge_date.';
COMMENT ON COLUMN claims.gold.claims_enriched.turnaround_days IS 'Days between discharge_date and settled_ts; NULL when the claim is not settled.';
COMMENT ON COLUMN claims.gold.claims_enriched.room_rent_over_cap IS 'TRUE if room_rent_per_day exceeds the plan cap room_rent_cap_per_day.';

In [0]:
%sql
-- Table and column documentation for claims.gold.hospital_monthly
COMMENT ON TABLE claims.gold.hospital_monthly IS
  'Gold aggregate (materialized view): monthly claims and financial summary per hospital, aggregated from claims.gold.claims_enriched.';

COMMENT ON COLUMN claims.gold.hospital_monthly.hospital_id IS 'Hospital identifier; join key to claims.raw.hospitals.';
COMMENT ON COLUMN claims.gold.hospital_monthly.admission_month IS 'First day of the admission month the metrics cover (yyyy-MM-01).';
COMMENT ON COLUMN claims.gold.hospital_monthly.claim_count IS 'Number of claims admitted at the hospital during the month.';
COMMENT ON COLUMN claims.gold.hospital_monthly.total_billed IS 'Sum of billed_amount for the hospital-month, in INR.';
COMMENT ON COLUMN claims.gold.hospital_monthly.total_approved IS 'Sum of approved_amount for the hospital-month, in INR.';
COMMENT ON COLUMN claims.gold.hospital_monthly.rejected_count IS 'Number of claims rejected at the hospital during the month.';

In [0]:
%sql
-- Table and column documentation for claims.gold.hospital_monthly_designer
COMMENT ON TABLE claims.gold.hospital_monthly_designer IS
  'Gold aggregate (materialized view): monthly claims and financial summary per hospital; variant of claims.gold.hospital_monthly built via visual data prep.';

COMMENT ON COLUMN claims.gold.hospital_monthly_designer.hospital_id IS 'Hospital identifier; join key to claims.raw.hospitals.';
COMMENT ON COLUMN claims.gold.hospital_monthly_designer.admission_month IS 'First day of the admission month the metrics cover (yyyy-MM-01).';
COMMENT ON COLUMN claims.gold.hospital_monthly_designer.claim_count IS 'Number of claims admitted at the hospital during the month.';
COMMENT ON COLUMN claims.gold.hospital_monthly_designer.total_billed IS 'Sum of billed_amount for the hospital-month, in INR.';
COMMENT ON COLUMN claims.gold.hospital_monthly_designer.total_approved IS 'Sum of approved_amount for the hospital-month, in INR.';
COMMENT ON COLUMN claims.gold.hospital_monthly_designer.rejected_count IS 'Number of claims rejected at the hospital during the month.';

In [0]:
%sql
-- Table and column documentation for claims.gold.hospital_monthly_vdp
COMMENT ON TABLE claims.gold.hospital_monthly_vdp IS
  'Gold aggregate (materialized view): monthly claims and financial summary per hospital; variant of claims.gold.hospital_monthly built via visual data prep.';

COMMENT ON COLUMN claims.gold.hospital_monthly_vdp.hospital_id IS 'Hospital identifier; join key to claims.raw.hospitals.';
COMMENT ON COLUMN claims.gold.hospital_monthly_vdp.admission_month IS 'First day of the admission month the metrics cover (yyyy-MM-01).';
COMMENT ON COLUMN claims.gold.hospital_monthly_vdp.claim_count IS 'Number of claims admitted at the hospital during the month.';
COMMENT ON COLUMN claims.gold.hospital_monthly_vdp.total_billed IS 'Sum of billed_amount for the hospital-month, in INR.';
COMMENT ON COLUMN claims.gold.hospital_monthly_vdp.total_approved IS 'Sum of approved_amount for the hospital-month, in INR.';
COMMENT ON COLUMN claims.gold.hospital_monthly_vdp.rejected_count IS 'Number of claims rejected at the hospital during the month.';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS claims.gov.audit_claims_enriched (
    version BIGINT,
    changed_at TIMESTAMP,
    changed_by STRING,
    operation STRING,
    source STRING
);

In [0]:
%sql
SELECT
    event_time,
    user_identity.email AS changed_by,
    service_name,
    action_name,
    request_params
FROM system.access.audit
WHERE TO_JSON(request_params)
      LIKE '%claims.gold.claims_enriched%'
ORDER BY event_time DESC
LIMIT 20;

In [0]:
from pyspark.sql.functions import (
    col, row_number, lit
)
from pyspark.sql.window import Window

audit_source = spark.sql("""
    SELECT
        event_time,
        user_identity.email AS changed_by,
        action_name AS operation
    FROM system.access.audit
    WHERE TO_JSON(request_params)
          LIKE '%claims.gold.claims_enriched%'
""")

window_spec = Window.orderBy(col("event_time").desc())

audit_df = (
    audit_source
    .withColumn(
        "version",
        row_number().over(window_spec).cast("BIGINT")
    )
    .limit(3)
    .select(
        "version",
        col("event_time").alias("changed_at"),
        "changed_by",
        "operation",
        lit("system.access.audit").alias("source")
    )
)

audit_df.write.mode("append").saveAsTable(
    "claims.gov.audit_claims_enriched"
)